In [1]:
from typesafe_sdk import TypeSafeClient, Choice

client = TypeSafeClient(model="jev-1.13.0")

print("JEV 客服分流实验初始化完成")

JEV 客服分流实验初始化完成


In [2]:
# 各类别对应的处理部门
ROUTES = {
    "refund": "退款处理组",
    "order_status": "订单查询组",
    "product_question": "产品咨询组",
    "account_issue": "账户支持组",
    "complaint": "投诉处理组",
    "other": "人工客服"
}

# 测试消息：expected 是我们预先设定的参考类别
TEST_CASES = [
    {
        "id": "T1",
        "message": "我的订单 A104 被扣了两次钱，想退回重复扣款。",
        "expected": "refund"
    },
    {
        "id": "T2",
        "message": "我的订单 A105 发货了吗？物流到哪里了？",
        "expected": "order_status"
    },
    {
        "id": "T3",
        "message": "X 型号耳机支持蓝牙 5.3 吗？",
        "expected": "product_question"
    },
    {
        "id": "T4",
        "message": "我一直登录不上账号，重置密码也失败了。",
        "expected": "account_issue"
    },
    {
        "id": "T5",
        "message": "你们客服一直不回复，我要投诉并升级处理。",
        "expected": "complaint"
    },
    {
        "id": "T6",
        "message": "请帮我播放一首音乐。",
        "expected": "other"
    },
    {
        "id": "T7",
        "message": "这个事情到底怎么处理？帮我看看。",
        "expected": "other"
    }
]

# 定义 JEV 的分类问题
ROUTER_QUESTIONS = {
    "intent": Choice(
        instructions=(
            "判断用户消息的主要诉求，并选择最合适的类别。"
            "只根据消息中表达的意图分类；如果信息不足，"
            "或者不属于其他明确类别，请选择 other。"
        ),
        criteria={
            "refund": "退款、退货退款、重复扣款或退款进度",
            "order_status": "订单状态、物流、发货或配送进度",
            "product_question": "产品功能、规格或使用方法咨询",
            "account_issue": "登录、密码或账户异常",
            "complaint": "明确投诉、不满或要求升级处理",
            "other": "不属于以上类别，或无法明确判断主要诉求"
        }
    )
}

print("测试消息数量：", len(TEST_CASES))
print("分类类别数量：", len(ROUTES))

测试消息数量： 7
分类类别数量： 6


In [3]:
records = []

for case in TEST_CASES:
    result = client.system_one(
        state={
            "message": case["message"]
        },
        questions=ROUTER_QUESTIONS
    )

    answer = result.answers["intent"]

    record = {
        "id": case["id"],
        "message": case["message"],
        "expected": case["expected"],
        "predicted": answer.choice,
        "confidence": answer.confidence,
        "probabilities": answer.probabilities
    }

    records.append(record)

    print("=" * 60)
    print("测试编号：", record["id"])
    print("用户消息：", record["message"])
    print("预期类别：", record["expected"])
    print("模型类别：", record["predicted"])
    print("置信度：", round(record["confidence"], 4))
    print("概率分布：", record["probabilities"])
    print(
        "是否符合预期：",
        record["predicted"] == record["expected"]
    )

测试编号： T1
用户消息： 我的订单 A104 被扣了两次钱，想退回重复扣款。
预期类别： refund
模型类别： refund
置信度： 1.0
概率分布： {'refund': 1.0, 'order_status': 0.0, 'product_question': 0.0, 'account_issue': 0.0, 'complaint': 0.0, 'other': 0.0}
是否符合预期： True
测试编号： T2
用户消息： 我的订单 A105 发货了吗？物流到哪里了？
预期类别： order_status
模型类别： order_status
置信度： 1.0
概率分布： {'refund': 0.0, 'order_status': 1.0, 'product_question': 0.0, 'account_issue': 0.0, 'complaint': 0.0, 'other': 0.0}
是否符合预期： True
测试编号： T3
用户消息： X 型号耳机支持蓝牙 5.3 吗？
预期类别： product_question
模型类别： product_question
置信度： 1.0
概率分布： {'refund': 0.0, 'order_status': 0.0, 'product_question': 1.0, 'account_issue': 0.0, 'complaint': 0.0, 'other': 0.0}
是否符合预期： True
测试编号： T4
用户消息： 我一直登录不上账号，重置密码也失败了。
预期类别： account_issue
模型类别： account_issue
置信度： 1.0
概率分布： {'refund': 0.0, 'order_status': 0.0, 'product_question': 0.0, 'account_issue': 1.0, 'complaint': 0.0, 'other': 0.0}
是否符合预期： True
测试编号： T5
用户消息： 你们客服一直不回复，我要投诉并升级处理。
预期类别： complaint
模型类别： complaint
置信度： 1.0
概率分布： {'refund': 0.0, 'order_status': 0.0, 'produc

In [4]:
def route_by_confidence(record, threshold):
    predicted = record["predicted"]
    confidence = record["confidence"]

    if predicted == "other":
        return "转人工：其他问题或意图不明确"

    if confidence >= threshold:
        return "自动分流：" + ROUTES[predicted]

    return "转人工：置信度不足"


# 使用相同的模型结果，比较三个阈值
for threshold in [0.60, 0.75, 0.90]:
    print("\n" + "=" * 65)
    print(f"置信度阈值：{threshold:.2f}")
    print("=" * 65)

    for record in records:
        decision = route_by_confidence(record, threshold)

        print(
            f"{record['id']} | "
            f"预期={record['expected']} | "
            f"预测={record['predicted']} | "
            f"置信度={record['confidence']:.3f} | "
            f"{decision}"
        )


置信度阈值：0.60
T1 | 预期=refund | 预测=refund | 置信度=1.000 | 自动分流：退款处理组
T2 | 预期=order_status | 预测=order_status | 置信度=1.000 | 自动分流：订单查询组
T3 | 预期=product_question | 预测=product_question | 置信度=1.000 | 自动分流：产品咨询组
T4 | 预期=account_issue | 预测=account_issue | 置信度=1.000 | 自动分流：账户支持组
T5 | 预期=complaint | 预测=complaint | 置信度=1.000 | 自动分流：投诉处理组
T6 | 预期=other | 预测=other | 置信度=1.000 | 转人工：其他问题或意图不明确
T7 | 预期=other | 预测=other | 置信度=0.960 | 转人工：其他问题或意图不明确

置信度阈值：0.75
T1 | 预期=refund | 预测=refund | 置信度=1.000 | 自动分流：退款处理组
T2 | 预期=order_status | 预测=order_status | 置信度=1.000 | 自动分流：订单查询组
T3 | 预期=product_question | 预测=product_question | 置信度=1.000 | 自动分流：产品咨询组
T4 | 预期=account_issue | 预测=account_issue | 置信度=1.000 | 自动分流：账户支持组
T5 | 预期=complaint | 预测=complaint | 置信度=1.000 | 自动分流：投诉处理组
T6 | 预期=other | 预测=other | 置信度=1.000 | 转人工：其他问题或意图不明确
T7 | 预期=other | 预测=other | 置信度=0.960 | 转人工：其他问题或意图不明确

置信度阈值：0.90
T1 | 预期=refund | 预测=refund | 置信度=1.000 | 自动分流：退款处理组
T2 | 预期=order_status | 预测=order_status | 置信度=1.000 | 自动分流：订单查询组
T3 | 预期=

In [5]:
# 新增更有挑战性的测试消息
EXTRA_CASES = [
    {
        "id": "T8",
        "message": "我账号登录不上，订单也查不了，退款进度也看不到。",
        "expected": "account_issue"
    },
    {
        "id": "T9",
        "message": "订单一直没到，我想申请退款，但也想先查清楚物流状态。",
        "expected": "other"
    },
    {
        "id": "T10",
        "message": "你们一直不处理我的退款，我要投诉客服。",
        "expected": "complaint"
    },
    {
        "id": "T11",
        "message": "这个问题麻烦处理一下，我也说不清是订单、账户还是退款的问题。",
        "expected": "other"
    },
    {
        "id": "T12",
        "message": "我买的耳机有杂音，想退货或者换货，应该找哪个部门？",
        "expected": "other"
    },
]

extra_records = []

for case in EXTRA_CASES:
    result = client.system_one(
        state={"message": case["message"]},
        questions=ROUTER_QUESTIONS
    )

    answer = result.answers["intent"]

    record = {
        "id": case["id"],
        "message": case["message"],
        "expected": case["expected"],
        "predicted": answer.choice,
        "confidence": answer.confidence,
        "probabilities": answer.probabilities
    }

    extra_records.append(record)

    print("=" * 60)
    print("编号：", record["id"])
    print("消息：", record["message"])
    print("参考类别：", record["expected"])
    print("预测类别：", record["predicted"])
    print("置信度：", round(record["confidence"], 4))
    print("概率分布：", record["probabilities"])
    print("是否符合参考类别：", record["predicted"] == record["expected"])

编号： T8
消息： 我账号登录不上，订单也查不了，退款进度也看不到。
参考类别： account_issue
预测类别： account_issue
置信度： 0.94
概率分布： {'refund': 0.0, 'order_status': 0.0, 'product_question': 0.0, 'account_issue': 0.95, 'complaint': 0.0, 'other': 0.05}
是否符合参考类别： True
编号： T9
消息： 订单一直没到，我想申请退款，但也想先查清楚物流状态。
参考类别： other
预测类别： order_status
置信度： 0.48
概率分布： {'refund': 0.41, 'order_status': 0.57, 'product_question': 0.0, 'account_issue': 0.0, 'complaint': 0.0, 'other': 0.02}
是否符合参考类别： False
编号： T10
消息： 你们一直不处理我的退款，我要投诉客服。
参考类别： complaint
预测类别： complaint
置信度： 0.76
概率分布： {'refund': 0.2, 'order_status': 0.0, 'product_question': 0.0, 'account_issue': 0.0, 'complaint': 0.8, 'other': 0.0}
是否符合参考类别： True
编号： T11
消息： 这个问题麻烦处理一下，我也说不清是订单、账户还是退款的问题。
参考类别： other
预测类别： other
置信度： 0.93
概率分布： {'refund': 0.0, 'order_status': 0.0, 'product_question': 0.0, 'account_issue': 0.0, 'complaint': 0.06, 'other': 0.9400000000000001}
是否符合参考类别： True
编号： T12
消息： 我买的耳机有杂音，想退货或者换货，应该找哪个部门？
参考类别： other
预测类别： refund
置信度： 0.78
概率分布： {'refund': 0.82, 'order_status': 0.

In [6]:
# 合并已有结果和新增结果，不重新调用 API
all_records = records + extra_records

for threshold in [0.60, 0.75, 0.90]:
    outcomes = [
        (record, route_by_confidence(record, threshold))
        for record in all_records
    ]

    # 自动分流的样本
    auto_rows = [
        (record, decision)
        for record, decision in outcomes
        if decision.startswith("自动分流：")
    ]

    manual_count = len(all_records) - len(auto_rows)

    # 全部样本的分类准确率（相对于人工设定的参考标签）
    overall_accuracy = sum(
        record["predicted"] == record["expected"]
        for record in all_records
    ) / len(all_records)

    # 仅统计自动分流样本的分类准确率
    auto_accuracy = (
        sum(
            record["predicted"] == record["expected"]
            for record, _ in auto_rows
        ) / len(auto_rows)
        if auto_rows else 0.0
    )

    print(f"\n===== 阈值：{threshold:.2f} =====")
    print(f"总体分类正确率：{overall_accuracy:.2%}")
    print(f"自动分流数量：{len(auto_rows)}/{len(all_records)}")
    print(f"自动分流率：{len(auto_rows) / len(all_records):.2%}")
    print(f"自动分流样本分类正确率：{auto_accuracy:.2%}")
    print(f"转人工数量：{manual_count}/{len(all_records)}")


===== 阈值：0.60 =====
总体分类正确率：83.33%
自动分流数量：8/12
自动分流率：66.67%
自动分流样本分类正确率：87.50%
转人工数量：4/12

===== 阈值：0.75 =====
总体分类正确率：83.33%
自动分流数量：8/12
自动分流率：66.67%
自动分流样本分类正确率：87.50%
转人工数量：4/12

===== 阈值：0.90 =====
总体分类正确率：83.33%
自动分流数量：6/12
自动分流率：50.00%
自动分流样本分类正确率：100.00%
转人工数量：6/12


In [7]:
# 比较阈值 0.75 和 0.90 下的分流变化

for record in all_records:
    decision_075 = route_by_confidence(record, 0.75)
    decision_090 = route_by_confidence(record, 0.90)

    if decision_075 != decision_090:
        print("=" * 50)
        print("测试编号：", record["id"])
        print("用户消息：", record["message"])
        print("预测类别：", record["predicted"])
        print("置信度：", round(record["confidence"], 4))
        print("阈值 0.75：", decision_075)
        print("阈值 0.90：", decision_090)

测试编号： T10
用户消息： 你们一直不处理我的退款，我要投诉客服。
预测类别： complaint
置信度： 0.76
阈值 0.75： 自动分流：投诉处理组
阈值 0.90： 转人工：置信度不足
测试编号： T12
用户消息： 我买的耳机有杂音，想退货或者换货，应该找哪个部门？
预测类别： refund
置信度： 0.78
阈值 0.75： 自动分流：退款处理组
阈值 0.90： 转人工：置信度不足


### 实验一：基于置信度的客服工单分类与分流

**实验目的：** 理解置信度门控的作用，分析不同置信度阈值对自动分流率和自动分流样本分类正确率的影响。

**实验方法：** 使用 JEV 对客服消息进行意图分类，记录预测类别、置信度和概率分布；随后设置 0.60、0.75 和 0.90 三个阈值，比较不同阈值下的分流结果。

**实验结果：** 在当前 12 条测试消息中，三个阈值下的总体分类正确率均为 83.33%。当阈值为 0.60 或 0.75 时，自动分流率为 66.67%，自动分流样本分类正确率为 87.50%；当阈值提高到 0.90 时，自动分流率下降至 50.00%，自动分流样本分类正确率达到 100%。

**实验分析：** 提高置信度阈值可以在当前测试集上减少错误的自动分流，但也会增加人工审核数量。置信度门控不能保证所有自动分流结果正确，也不能解决模型对错误结果过度自信的问题。此外，测试消息的参考标签需要具有明确、统一的业务定义。

**个人收获：** 我理解了分类与分流是两个不同的环节。模型负责预测意图，程序根据置信度和业务规则决定是否自动处理。实际应用中，需要根据错误成本、人工审核能力和业务风险选择合适的门控策略，而不能只追求较高的自动分流率。


In [15]:
#实验二：检索与证据——基于客服政策的回答核查
# 1. 建立模拟客服政策知识库
POLICY_KB = [
    {
        "id": "P1",
        "title": "退款到账时效",
        "text": (
            "退款申请审核通过后，款项通常会在3—5个工作日内"
            "按原支付渠道退回；实际到账时间以支付渠道处理进度为准。"
            "本政策不承诺24小时内到账。"
        ),
        "keywords": [
            "退款多久到账", "几天到账", "几天内到账",
            "到账时间", "24小时内到账", "工作日",
            "当天到账", "多久退回"
        ]
    },
    {
        "id": "P2",
        "title": "重复扣款处理",
        "text": (
            "经核实确实存在重复扣款的，用户提供订单号及支付记录后，"
            "可退还重复扣款的金额。"
        ),
        "keywords": [
            "重复扣款", "扣了两次", "扣款两次",
            "重复支付", "重复付款"
        ]
    },
    {
        "id": "P3",
        "title": "数字商品退款政策",
        "text": (
            "数字商品或服务开通后，原则上不支持因个人不喜欢"
            "而申请无理由退款；如存在重复扣款或系统故障，"
            "应核实具体情况后处理。"
        ),
        "keywords": [
            "数字商品", "无理由退款", "不喜欢", "开通后"
        ]
    }
]

print("知识库条款数量：", len(POLICY_KB))

for policy in POLICY_KB:
    print(f'{policy["id"]}：{policy["title"]}')
    print("政策内容：", policy["text"])
    print("-" * 50)

知识库条款数量： 3
P1：退款到账时效
政策内容： 退款申请审核通过后，款项通常会在3—5个工作日内按原支付渠道退回；实际到账时间以支付渠道处理进度为准。本政策不承诺24小时内到账。
--------------------------------------------------
P2：重复扣款处理
政策内容： 经核实确实存在重复扣款的，用户提供订单号及支付记录后，可退还重复扣款的金额。
--------------------------------------------------
P3：数字商品退款政策
政策内容： 数字商品或服务开通后，原则上不支持因个人不喜欢而申请无理由退款；如存在重复扣款或系统故障，应核实具体情况后处理。
--------------------------------------------------


In [16]:
def retrieve_policy(question):
    """根据问题中的关键词，检索最相关的政策条款。"""
    candidates = []

    for policy in POLICY_KB:
        score = sum(
            1 for keyword in policy["keywords"]
            if keyword in question
        )

        if score > 0:
            candidates.append((score, policy))

    if not candidates:
        return None

    # 匹配关键词数量最多的条款优先
    candidates.sort(key=lambda item: item[0], reverse=True)

    return candidates[0][1]


# 先单独测试检索器
test_questions = [
    "退款通常几天到账？",
    "我的订单被扣了两次，可以退一笔吗？",
    "数字商品开通后，因为不喜欢可以无理由退款吗？",
    "退款到银行卡会不会收手续费？"
]

for question in test_questions:
    policy = retrieve_policy(question)

    print("=" * 50)
    print("用户问题：", question)

    if policy is None:
        print("检索结果：未找到相关政策")
    else:
        print("检索到：", policy["id"], policy["title"])
        print("政策内容：", policy["text"])

用户问题： 退款通常几天到账？
检索到： P1 退款到账时效
政策内容： 退款申请审核通过后，款项通常会在3—5个工作日内按原支付渠道退回；实际到账时间以支付渠道处理进度为准。本政策不承诺24小时内到账。
用户问题： 我的订单被扣了两次，可以退一笔吗？
检索到： P2 重复扣款处理
政策内容： 经核实确实存在重复扣款的，用户提供订单号及支付记录后，可退还重复扣款的金额。
用户问题： 数字商品开通后，因为不喜欢可以无理由退款吗？
检索到： P3 数字商品退款政策
政策内容： 数字商品或服务开通后，原则上不支持因个人不喜欢而申请无理由退款；如存在重复扣款或系统故障，应核实具体情况后处理。
用户问题： 退款到银行卡会不会收手续费？
检索结果：未找到相关政策


In [17]:
from typesafe_sdk import Choice

# 三个证据核查案例
EVIDENCE_CASES = [
    {
        "id": "E1",
        "question": "退款通常几天到账？",
        "candidate_answer": (
            "退款审核通过后，通常需要3—5个工作日，"
            "实际到账以支付渠道处理进度为准。"
        ),
        "expected": "supports"
    },
    {
        "id": "E2",
        "question": "退款通常几天到账？",
        "candidate_answer": "所有退款都保证在24小时内到账。",
        "expected": "contradicts"
    },
    {
        "id": "E3",
        "question": "我的订单被扣了两次，可以退一笔吗？",
        "candidate_answer": "重复扣款的退款一定会在24小时内到账。",
        "expected": "insufficient"
    },
    {
        "id": "E4",
        "question": "退款到银行卡会不会收手续费？",
        "candidate_answer": "退款到银行卡一定不收任何手续费。",
        "expected": "no_evidence"
    }
]

EVIDENCE_THRESHOLD = 0.80

evidence_records = []

for case in EVIDENCE_CASES:

    # 第一步：检索证据
    policy = retrieve_policy(case["question"])

    print("=" * 60)
    print("案例编号：", case["id"])
    print("用户问题：", case["question"])
    print("候选回答：", case["candidate_answer"])

    # 第二步：检查是否检索到证据
    if policy is None:
        print("核查结果：未检索到相关政策，不能直接回答")
        evidence_records.append({
            "id": case["id"],
            "expected": case["expected"],
            "relation": "no_evidence",
            "confidence": None,
            "decision": "补充检索或转人工"
        })
        print("是否符合预期：", "no_evidence" == case["expected"])
        continue

    print("证据来源：", policy["id"], policy["title"])
    print("检索证据：", policy["text"])

    # 第三步：让 JEV 判断证据与候选回答的关系
    state = {
        "question": case["question"],
        "candidate_answer": case["candidate_answer"],
        "evidence": policy["text"],
        "evidence_id": policy["id"]
    }

    result = client.system_one(
        state=state,
        questions={
            "relation": Choice(
                instructions=(
                    "判断给定证据与候选回答的核心主张之间的关系。"
                    "只能根据提供的证据判断，不能自行补充外部事实。"
                    "如果证据明确支持核心主张，选择 supports；"
                    "如果证据明确与核心主张矛盾，选择 contradicts；"
                    "如果证据不足以判断核心主张，选择 insufficient。"
                ),
                criteria={
                    "supports": "证据足以支持候选回答的核心主张",
                    "contradicts": "证据明确与候选回答的核心主张矛盾",
                    "insufficient": "证据不足，无法确认或否定核心主张"
                }
            )
        }
    )

    answer = result.answers["relation"]
    relation = answer.choice
    confidence = answer.confidence

    # 第四步：由 Python 根据判断结果和置信度决定下一步
    if relation == "supports" and confidence >= EVIDENCE_THRESHOLD:
        decision = "允许依据证据回答，但不得超出证据范围"
    elif relation == "contradicts" and confidence >= EVIDENCE_THRESHOLD:
        decision = "拒绝当前候选回答，依据证据纠正"
    else:
        decision = "暂不自动回答，补充检索或转人工"

    record = {
        "id": case["id"],
        "expected": case["expected"],
        "relation": relation,
        "confidence": confidence,
        "decision": decision
    }
    evidence_records.append(record)

    print("JEV 判断：", relation)
    print("判断置信度：", round(confidence, 4))
    print("关系概率分布：", answer.probabilities)
    print("程序决策：", decision)
    print("是否符合预期：", relation == case["expected"])

print("\n实验二：证据核查汇总")
for record in evidence_records:
    print(record)

案例编号： E1
用户问题： 退款通常几天到账？
候选回答： 退款审核通过后，通常需要3—5个工作日，实际到账以支付渠道处理进度为准。
证据来源： P1 退款到账时效
检索证据： 退款申请审核通过后，款项通常会在3—5个工作日内按原支付渠道退回；实际到账时间以支付渠道处理进度为准。本政策不承诺24小时内到账。
JEV 判断： supports
判断置信度： 1.0
关系概率分布： {'supports': 1.0, 'contradicts': 0.0, 'insufficient': 0.0}
程序决策： 允许依据证据回答，但不得超出证据范围
是否符合预期： True
案例编号： E2
用户问题： 退款通常几天到账？
候选回答： 所有退款都保证在24小时内到账。
证据来源： P1 退款到账时效
检索证据： 退款申请审核通过后，款项通常会在3—5个工作日内按原支付渠道退回；实际到账时间以支付渠道处理进度为准。本政策不承诺24小时内到账。
JEV 判断： contradicts
判断置信度： 1.0
关系概率分布： {'supports': 0.0, 'contradicts': 1.0, 'insufficient': 0.0}
程序决策： 拒绝当前候选回答，依据证据纠正
是否符合预期： True
案例编号： E3
用户问题： 我的订单被扣了两次，可以退一笔吗？
候选回答： 重复扣款的退款一定会在24小时内到账。
证据来源： P2 重复扣款处理
检索证据： 经核实确实存在重复扣款的，用户提供订单号及支付记录后，可退还重复扣款的金额。
JEV 判断： insufficient
判断置信度： 1.0
关系概率分布： {'supports': 0.0, 'contradicts': 0.0, 'insufficient': 1.0}
程序决策： 暂不自动回答，补充检索或转人工
是否符合预期： True
案例编号： E4
用户问题： 退款到银行卡会不会收手续费？
候选回答： 退款到银行卡一定不收任何手续费。
核查结果：未检索到相关政策，不能直接回答
是否符合预期： True

实验二：证据核查汇总
{'id': 'E1', 'expected': 'supports', 'relation': 'supports', 'confidence': 1.0, 'decision

In [18]:
print("===== 实验二自动评估 =====")

correct = sum(
    record["relation"] == record["expected"]
    for record in evidence_records
)

total = len(evidence_records)

print(f"符合预期的案例：{correct}/{total}")
print(f"案例匹配率：{correct / total:.2%}")

for record in evidence_records:
    matched = record["relation"] == record["expected"]

    print(
        f'{record["id"]}: '
        f'预期={record["expected"]}, '
        f'实际={record["relation"]}, '
        f'匹配={matched}'
    )

===== 实验二自动评估 =====
符合预期的案例：4/4
案例匹配率：100.00%
E1: 预期=supports, 实际=supports, 匹配=True
E2: 预期=contradicts, 实际=contradicts, 匹配=True
E3: 预期=insufficient, 实际=insufficient, 匹配=True
E4: 预期=no_evidence, 实际=no_evidence, 匹配=True


# Task 4｜实验二学习心得：检索与证据核查

通过本次实验，我构建了一个基于客服退款政策的证据核查流程，实现了从政策检索、证据判断到程序决策的基本闭环。以下是我的一些理解：

1. **核心流程理解**：首先建立包含退款到账时效、重复扣款处理和数字商品退款规则的政策知识库，再根据用户问题检索相关政策，最后使用 JEV 判断候选回答与证据之间的关系。

2. **证据关系判断**：本次实验将证据关系划分为 `supports`（支持）、`contradicts`（矛盾）和 `insufficient`（证据不足）。如果没有检索到相关政策，则直接进入补充检索或人工处理流程，不继续调用 JEV。例如，政策说明退款通常需要 3—5 个工作日，可以支持一般性的到账时间回答，但不能支持“保证 24 小时内到账”的绝对承诺。

3. **置信度与程序决策**：JEV 输出判断结果和置信度，Python 再根据判断结果及设定的阈值决定下一步操作。置信度较高不代表判断一定正确，因此还需要结合证据内容和业务规则进行判断。

4. **实验发现与局限**：四个测试案例的实际判断均符合预期。但这只能说明当前系统在这组测试数据上运行正常，不能证明其在真实业务中具有同样的准确率。此外，当前政策检索主要依赖关键词匹配，可能遗漏使用不同表达方式描述的相关政策，后续可以尝试引入语义检索和更多测试案例。

通过本次实验，我认识到：**检索到相关内容不等于获得了有效证据，模型生成了判断也不等于事实已经核实。** 可靠的客服助手需要将证据检索、语义核查和程序决策结合起来，并为证据不足或判断不确定的情况保留人工处理路径。